## Parameters API LLM

### Install and import libraries

In [1]:
# (setup cell already installs what this notebook needs)

from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

False

Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

### Helper function requesting LLM

In [ ]:
def test_generation(param_name, values, prompt="Write a product description for a cordless barcode scanner."):
    for v in values:
        print("="*60)
        print(f"{param_name} = {v}")
        llm = make_llm(temperature=0.7)  # bazowe settings
        kwargs = {param_name: v}
        response = llm.invoke(prompt, **kwargs)
        print(response.content, "\n")

### Experiment with temperature
Temperature reshapes the probabilities : it flattens the distribution (high temp, making unlikely words relatively more likely) or sharpens it (low temp, concentrating mass on the top words). It rescales the odds but keeps every token in play

In [3]:
test_generation("temperature", [0.0, 0.7, 1.2])

temperature = 0.0
Recipe Name: Tropical Coconut Mango Cheesecake

Ingredients:

For the crust:
1. 1 1/2 cups graham cracker crumbs
2. 1/4 cup granulated sugar
3. 1/2 cup unsalted butter, melted

For the filling:
1. 3 (8 oz) packages cream cheese, softened
2. 1 cup granulated sugar
3. 3 large eggs
4. 1 cup canned coconut milk
5. 1 teaspoon pure vanilla extract
6. 1 tablespoon coconut extract
7. Zest and juice of 1 lime
8. 1 ripe mango, peeled and pureed

For the topping:
1. 1 cup whipped cream
2. 1 ripe mango, peeled and diced
3. 1/2 cup toasted shredded coconut

Instructions:

1. Preheat your oven to 325°F (163°C). 

2. In a medium bowl, combine the graham cracker crumbs, sugar, and melted butter. Press the mixture into the bottom of a 9-inch springform pan. Bake for 10 minutes, then remove from the oven and set aside.

3. In a large bowl, beat the cream cheese until smooth. Gradually add the sugar, beating until well combined. Add the eggs, one at a time, beating well after each addit

### Experiment with top_p
top_p controls how many of the model's candidate next-tokens are allowed to be considered when generating each word ; it's a way of trimming the model's vocabulary to the most probable options at each step. It's also called nucleus sampling, and it works alongside temperature as the two main levers on randomness.
Here's the mechanism. At every step, the model produces a probability for every possible next token. top_p says: sort those tokens from most to least likely, then keep only the smallest set whose probabilities add up to p, and sample the next token from just that set. The rest are discarded for that step.
So the value is a cumulative-probability threshold:

top_p = 1.0 : keep everything; the full distribution is in play (maximum diversity).
top_p = 0.9 : keep only the most likely tokens that together make up 90% of the probability mass; the long tail of unlikely words is cut off.
top_p = 0.1 : keep only the tiny set of tokens covering the top 10%; the model becomes very focused and repetitive, almost always picking the obvious word.

top_p truncates the distribution : it removes the unlikely tail entirely, then samples from what remains. It's a hard cutoff

In [4]:
test_generation("top_p", [0.2, 0.7, 1.0])


top_p = 0.2
Recipe Name: Tropical Bliss Cheesecake

Ingredients:

For the crust:
1. 1 1/2 cups graham cracker crumbs
2. 1/4 cup melted unsalted butter
3. 1/4 cup granulated sugar
4. 1/2 teaspoon cinnamon

For the filling:
1. 3 (8 oz) packages cream cheese, softened
2. 1 cup granulated sugar
3. 3 large eggs
4. 1 cup canned coconut milk
5. 1 tablespoon lime zest
6. 1 teaspoon vanilla extract

For the topping:
1. 1 cup diced fresh pineapple
2. 1 cup diced fresh mango
3. 1/2 cup shredded coconut
4. 1/4 cup honey
5. 1 tablespoon lime juice

Instructions:

1. Preheat your oven to 325°F (163°C). 

2. In a medium bowl, combine the graham cracker crumbs, melted butter, sugar, and cinnamon. Press the mixture into the bottom of a 9-inch springform pan to form the crust.

3. In a large bowl, beat the cream cheese until smooth. Gradually add the sugar, beating until well combined. Add the eggs, one at a time, beating well after each addition. Stir in the coconut milk, lime zest, and vanilla extract

### Experiment with max_tokens
max_tokens sets the maximum length of the model's response, the ceiling on how many tokens it's allowed to generate in its reply. When the model hits that limit, generation stops, even if it was mid-sentence. It's a length cap on the output, nothing more.

In [ ]:
test_generation("max_tokens", [30, 100, 300])


max_tokens = 30
Title: Lavender Honey Cheesecake

Ingredients:
For the Crust:
- 2 cups graham cracker crumbs
- 1/ 

max_tokens = 100
Title: Tropical Pineapple Coconut Cheesecake

Ingredients:

For the Crust:
- 1 1/2 cups graham cracker crumbs
- 1/2 cup sweetened shredded coconut
- 1/4 cup granulated sugar
- 1/2 cup unsalted butter, melted

For the Filling:
- 3 (8 oz) packages of cream cheese, softened
- 1 1/4 cup granulated sugar
- 3 large eggs 

max_tokens = 300


💡 Effect:

When temperature=0, the responses will be repetitive,
When temperature=1.2, the responses will be more frantic,
When top_p is low, the model will be conservative,
When top_k is high, the model will be more diverse,
max_tokens will decide whether the story is 2 sentences or a whole page.